# Plotting Final — Full-Series Historical State Plot

Purpose: select a series and historical month/year. Plot the **entire available series**, while highlighting the selected month (default: May 1995). The selected month is an event marker, not a filter that truncates the dataset.

Panel 1 standardizes every observation using the full-series mean and sample standard deviation. Panel 2 shows the full raw series with trailing rolling means of 2 and 4 observations. Because full-series statistics include observations after the target date, this is an exploratory retrospective visualization—not a no-look-ahead backtest. For real-time-style testing, use expanding or trailing statistics calculated only from past observations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Experiment controls ----
SERIES_ID = 'CPIAUCSL'  # Any CSV filename under data/cleaned, without .csv
TARGET_YEAR = 1995
TARGET_MONTH = 5
ROLLING_WINDOWS = (2, 4)

# Locate data regardless of whether Jupyter starts in repository root or notebook folder.
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p.resolve() for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Start Jupyter from repository root or notebook folder.')

csv_path = cleaned_dir / f'{SERIES_ID}.csv'
if not csv_path.exists():
    available = sorted(p.stem for p in cleaned_dir.glob('*.csv'))
    raise FileNotFoundError(f'{csv_path} not found. Available series include: {available[:20]}')

df = pd.read_csv(csv_path)
if not {'date', 'value'}.issubset(df.columns):
    raise ValueError(f'{SERIES_ID}.csv must contain date and value columns; found {list(df.columns)}')

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
series = (df.dropna(subset=['date', 'value'])
            .sort_values('date')
            .drop_duplicates('date', keep='last')
            .set_index('date')['value'])

if not 1 <= TARGET_MONTH <= 12:
    raise ValueError('TARGET_MONTH must be between 1 and 12.')
month_start = pd.Timestamp(TARGET_YEAR, TARGET_MONTH, 1)
month_end = month_start + pd.offsets.MonthEnd(1)
target_mask = (series.index >= month_start) & (series.index <= month_end)
if not target_mask.any():
    raise ValueError(f'No observations for {SERIES_ID} in {month_start:%B %Y}.')

# Full-series standardized values (retrospective, not point-in-time).
reference_mean = series.mean()
reference_std = series.std(ddof=1)
if not np.isfinite(reference_std) or reference_std == 0:
    raise ValueError('Full-series standard deviation is zero or undefined.')
series_z = (series - reference_mean) / reference_std

rolling_2 = series.rolling(window=ROLLING_WINDOWS[0], min_periods=ROLLING_WINDOWS[0]).mean()
rolling_4 = series.rolling(window=ROLLING_WINDOWS[1], min_periods=ROLLING_WINDOWS[1]).mean()

print(f'Series: {SERIES_ID} | Highlighted month: {month_start:%B %Y}')
print(f'Full data coverage: {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}')
print(f'Total observations: {len(series)} | Observations in target month: {int(target_mask.sum())}')
print(f'Full-series mean: {reference_mean:.6g} | Sample std: {reference_std:.6g}')
display(pd.DataFrame({
    'value': series,
    'full_series_z_score': series_z,
    'rolling_mean_2': rolling_2,
    'rolling_mean_4': rolling_4,
    'is_target_month': target_mask
}))


## Full-series plots

Both panels display every observation in the selected series. The vertical band marks the chosen month. The standardized panel uses the full-series mean and sample standard deviation; the raw panel overlays trailing 2- and 4-observation rolling means. Full-series standardization uses future observations relative to historical target dates, so interpret it descriptively rather than as a real-time signal.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, constrained_layout=True)

# Highlight the selected month in both full-series panels.
for ax in axes:
    ax.axvspan(month_start, month_end, color='orange', alpha=0.18, label=f'Target: {month_start:%B %Y}')

axes[0].plot(series_z.index, series_z.values, linewidth=1.1, label='Full-series z-score')
axes[0].axhline(0, color='black', linewidth=1, label='Full-series mean')
axes[0].axhline(1, linestyle='--', alpha=0.6, label='+1 std')
axes[0].axhline(-1, linestyle='--', alpha=0.6, label='-1 std')
axes[0].axhline(2, linestyle=':', alpha=0.6, label='+2 std')
axes[0].axhline(-2, linestyle=':', alpha=0.6, label='-2 std')
axes[0].set_title(f'{SERIES_ID} — full-series standardized values')
axes[0].set_ylabel('Z-score')
axes[0].legend(ncol=3, fontsize=8)
axes[0].grid(True, alpha=0.2)

axes[1].plot(series.index, series.values, linewidth=1.1, label='Observed value')
axes[1].plot(rolling_2.index, rolling_2.values, linewidth=1.3, label='Rolling mean (2 observations)')
axes[1].plot(rolling_4.index, rolling_4.values, linewidth=1.6, label='Rolling mean (4 observations)')
axes[1].set_title('Full observed series with trailing rolling means')
axes[1].set_ylabel('Original units')
axes[1].set_xlabel('Observation date')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.show()

print('Observations in highlighted month:')
display(series.loc[target_mask].rename('value').to_frame())


## Interpretation checklist

- Where does the selected month sit relative to the full history?
- Does the series show persistent trends, sharp shocks, or possible regime changes?
- How do the 2- and 4-observation rolling means differ around the highlighted date?
- Remember: full-series mean/std includes future observations. For a genuine point-in-time test, calculate the baseline using only data available before each date.

This notebook is a descriptive visualization, not evidence by itself of mean reversion or a tradable anomaly.